# 第17章 Hugging Faceと事前学習済みモデル

世界中のモデルを借りる前に知っておくこと：ベースとチャット、ライセンス、プロンプト

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：ベースモデルとチャットモデルの比較実験、複数ターンの会話、few-shotプロンプトの効果測定。Hugging Faceのアカウントとトークンも安全に準備する
- **身につく数学・理論**：事前学習→SFT→好みの学習（ベースとチャットの違い） ／ モデルカードとライセンス ／ チャットテンプレートと複数ターンの会話 ／ プロンプト技法（zero-shot・few-shot・段階的に考える）とその限界 ／ モデルの大きさとGPUメモリの見積もり ／ safetensors（安全な重みのファイル形式）
- **新しく使うPython・PyTorch**：`AutoModelForCausalLM` / `AutoTokenizer` ／ `apply_chat_template` ／ `generation_config` ／ Colabのシークレット（`userdata`）

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（GPU：T4） |
| 所要時間目安 | 90分 |
| 難易度 | ★★★☆☆ |

> **⚠️ GPUランタイム設定**：メニューの「ランタイム」→「ランタイムのタイプを変更」→「**T4 GPU**」を選んでから上から順に実行してください。
> 無料枠の混雑・上限でGPUに接続できない場合は、時間をおくか [Kaggle Notebooks](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html) を使うか、各章のCPU短縮手順で進められます。接続切断に備え、学習済みモデルはこまめにGoogleドライブへ保存しましょう。

> **🎯 この章でつくるもの**
>
> 第4部まで、あなたは「LLMを作る側」でした。ここからは、世界中の研究機関や企業が公開している**事前学習済みモデル**を借りて、自分の目的に合わせて鍛える側に回ります。この章ではその準備として、Hugging Faceでのモデルの選び方、ベースモデルとチャットモデルの違い、プロンプトの工夫とその限界を、実験しながら学びます。

> **🔗 これまでの復習とつながり：第1章〜第3章・第15章とのつながり**
>
> - **第1章**：`pipeline` で手軽にチャットを動かし、会話は `role`（`system` / `user` / `assistant`）の台本形式（`apply_chat_template`）で渡されることを見ました。
> - **第2章・第3章**：特殊トークン（`<|im_end|>`）・コンテキスト長の上限・`temperature` や `do_sample` といった生成設定（`generation_config`）の中身を学びました。
> - **第15章**：大きな事前学習は「借りる」のが正解であり、学習には1パラメータ約16バイトのVRAMが必要になることを見積もりました。
>
> この章では、その知識を土台にして、**モデルを借りる前のチェック項目（ライセンス・ベース vs チャット・履歴管理・プロンプトの限界・推論 vs 学習のVRAM）**を実務レベルで整理します。

**ミナト**：第1章でQwenを使ったときは、名前をコピペしただけでした。

**ボンゴ先生**：それで正解だった。でも、これからモデルを鍛えるなら、借りるモデルの「素性」を知っておく必要がある。どんなデータで育ったのか、何をしていいライセンスなのか、どれくらいの大きさなのか。中古車を買うときに、車検証を見るのと同じだ。

## Hugging Face Hub を歩く

**Hugging Face Hub**（<https://huggingface.co/>）は、LLMを含むAIモデルとデータセットの、世界最大の置き場です。GitHubのAI版と考えてください。ブラウザで開いて、検索窓に `Qwen2.5-0.5B-Instruct` と入れてみましょう。モデルのページ（**モデルカード**）には、次のような情報が書いてあります。

| 項目 | 何が分かる？ |
|---|---|
| 説明（Model card） | どんな目的で作られたか、どんなデータで学習したか、性能、使い方 |
| ライセンス（License） | 商用利用してよいか、改変して公開してよいか |
| Files and versions | 重みのファイル（`model.safetensors`）、設定（`config.json`）、トークナイザー |
| パラメータ数・テンソル型 | 大きさ（0.5Bなど）と、保存されている数の形式（bfloat16など） |

> **💡 理論のポイント：ライセンスは必ず読む**
>
> モデルにも、ソフトウェアと同じようにライセンス（利用条件）があります。
>
> - **Apache 2.0 / MIT**：商用利用も改変も自由（Qwen2.5の多くのサイズ、など）
> - **独自ライセンス**：Llama（Meta）やGemma（Google）は、独自の利用規約に同意する必要がある。条件付きで商用利用可
> - **CC BY-NC（非営利）**：研究や個人の学習はOKだが、商用は不可
>
> 学校の課題や個人の勉強ならほとんど問題ありませんが、**作ったものを公開したり、アルバイト先や就職先で使ったりする**ときには、必ず確認しましょう。学習に使ったデータセットのライセンスも同様です。

#### 📝 確認問題（`ch17-q01` · Lv.1 · モデルカードとライセンス（Apache 2.0 / MIT / CC BY-NC / 独自ライセンス））

Hugging Face Hub で公開されている事前学習済みモデルを**企業の商用サービス**に組み込んで使いたいとき、ライセンス欄の確認として最も適切な判断はどれですか？

- **a.** ベースモデルのライセンスさえ商用可であれば、追加学習に使うデータセットのライセンスは確認しなくてよい
- **b.** Apache 2.0やMITは商用利用可、CC BY-NCは非営利限定で商用不可、独自規約は個別条件を確認する
- **c.** CC BY-NCは「No Charge（無料商用可）」の略称であるため、企業の商用サービスへ組み込むのに最適である
- **d.** Hugging Face Hubで無料公開されているモデルはすべて商用利用が自由なため、ライセンス確認は不要である

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：オープンソース系の定番ライセンスと、Creative Commons の「NC」が何を意味するかを確認しましょう。
- **ヒント 2**：「NC」は Non-Commercial（非営利）の略です。さらにLlamaやGemmaなどの独自ライセンスは固有の利用条件（規約同意や規模制限等）を確認する必要があります。
- **正解**：`b`

**考え方**：モデルカードは中古車の車検証のようなものです。公開・商用利用の前には、モデルとデータセット両方のライセンス条件を必ず確認します。

1. Apache 2.0 / MIT：商用利用・改変ともに自由度が高い。
1. Llama / Gemmaなどの独自ライセンス：規約同意（gated）や売上規模などの独自条件を確認。
1. CC BY-NC：個人学習・研究は可だが商用不可。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q01

</details>

## ベースモデルとチャットモデル

> **✨ コツ：実行環境とリソースの目安（CPUでも実行可能）**
>
> - **環境**：無料GPU（Colab T4）推奨ですが、0.5Bモデルのため**CPUでも実行できます**。
> - **ダウンロードとメモリ**：ベースモデルとチャットモデルの2つを合わせて初回約2GBをダウンロードします。必要メモリはGPU（float16）でVRAM約2〜3GB、CPU（float32）でRAM約4〜5GBです。
> - **所要時間と代替手順**：GPUなら各生成は数秒、CPUでは1回あたり十数秒〜数十秒（レビュー6問×2通りは数分）かかります。GPUが借りられない日やRAMが厳しいときは、ベースモデルの実験が終わった後に `del base` してからチャットモデルを読み込むと軽く動かせます。
> - **ライブラリ互換性**：`dtype=` 引数は `transformers >= 4.56`（5系含む）に対応しています（古い版では `torch_dtype=` に読み替えてください）。また `device_map="auto"` には `accelerate` パッケージが必要です（Colabには標準導入済み。ローカル環境はリポジトリの `requirements-book.txt` を参照）。

Qwen2.5-0.5Bには、2つの版があります。`Qwen2.5-0.5B`（**ベースモデル**）と、`Qwen2.5-0.5B-Instruct`（**チャットモデル**、指示に従うモデル）です。同じ質問をして、違いを見てみましょう。GPUのランタイムで実行してください。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

gpu = torch.cuda.is_available()
dtype = torch.float16 if gpu else torch.float32

def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=dtype, device_map="auto")
    return tok, model

base_tok, base = load("Qwen/Qwen2.5-0.5B")
chat_tok, chat = load("Qwen/Qwen2.5-0.5B-Instruct")
print("読み込み完了")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
読み込み完了
```

</details>

> **🐍 Python・PyTorchのポイント：AutoModelForCausalLM って何？**
>
> `AutoModel` ではなく `AutoModelForCausalLM` という長い名前を使うのはなぜでしょう？
> `AutoModel` はTransformerの胴体部分（各トークンの特徴量ベクトルを出すところまで）しか読み込みません。一方、`AutoModelForCausalLM`（Causal Language Model：因果的言語モデル）は、その上に第12章や第15章で作った「次のトークンを予測する出力層（`lm_head`）」をちゃんと乗せたモデルを読み込んでくれます。文章の続きを生成したいときは、必ずこの `ForCausalLM` のほうを使います。

ベースモデルに、質問の文章をそのまま渡してみます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def complete(tok, model, text, n=60):
    inputs = tok(text, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(complete(base_tok, base, "日本で一番高い山は何ですか？"))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
日本で一番高い山は何ですか？
日本で一番高い山は何ですか？
日本で一番高い山は何ですか？
日本で一番高い山は何ですか？
```

</details>

**ボンゴ先生**：同じ質問をくり返している！（キミの画面では、答えを書いた後に、別の質問を勝手に書き続けているかもしれない。どちらも同じ理由だ。）ベースモデルは「質問に答える」ことを教わっていない。ネットで「質問文」の次によく来るのは……別の質問文（よくある質問の一覧ページなど）だからね。ベースモデルは、ひたすら**ネットの文章の続き**を書く。それだけだ。

答えさせたいなら、「答えが続きそうな文章」にしてあげればいいのです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(complete(base_tok, base, "問：日本で一番高い山は何ですか？\n答：", n=30))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
富士山です。富士山は標高3776メートルで、静岡県と山梨県にまたがっています。
問：日本で一番長い川は何ですか？
答：
```

</details>

答えた後、また次の「問」を勝手に作り始めました。止まるタイミングを知らないのです。一方、チャットモデルは……

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def ask(tok, model, messages, n=100):
    inputs = tok.apply_chat_template(messages, add_generation_prompt=True,
                                     return_tensors="pt", return_dict=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=n, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

print(ask(chat_tok, chat, [{"role": "user", "content": "日本で一番高い山は何ですか？"}]))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
日本で一番高い山は富士山です。標高は3,776メートルで、静岡県と山梨県の境にあります。
```

</details>

きちんと答えて、きちんと止まりました。

> **💡 理論のポイント：LLMができるまでの3段階**
>
> チャットモデルは、ふつう次の3段階で作られます。
>
> 1. **事前学習**（pre-training）：大量の文章で「次のトークン予測」を学ぶ（第13〜15章でやったこと）。ここでできるのが**ベースモデル**。知識と文章力は身につくが、会話の作法は知らない
> 2. **SFT**（教師ありファインチューニング）：「質問 → 良い答え」の例を数万〜数百万件、チャットテンプレート（第1章の台本形式）で学ばせる。会話の作法と、`<|im_end|>` で止まることを覚える（**第18章**）
> 3. **好みの学習**（RLHF・DPOなど）：同じ質問への2つの答えのどちらが良いかを人間が選び、良い方を出しやすくする。安全性や丁寧さもここで整える（**第22章**）
>
> ベースモデルは「知識豊富だけど空気が読めない人」、チャットモデルは「それに接客研修を受けさせた人」、というイメージです。

<details><summary><b>基礎の補足：ひと目で比較：Base（ベース）モデル vs Instruct（チャット）モデル</b>（クリックで開く）</summary>

構造もパラメータ数（0.5B）もまったく同じですが、追加学習の有無で使い方が大きく変わります。

| 比較項目 | Baseモデル（例：`Qwen2.5-0.5B`） | Instruct / Chatモデル（例：`Qwen2.5-0.5B-Instruct`） |
|---|---|---|
| 完了した学習段階 | ①事前学習のみ | ①事前学習 → ②SFT → ③好みの学習（RLHF / DPO） |
| 得意な仕事 | 文章の続きを補完する・独自SFT（第18章）の土台にする | 質問への回答・指示遂行・対話アプリ |
| 入力の渡し方 | 生のテキスト（`"問：…\n答："` など） | `messages`＋`apply_chat_template(...)` |
| 止まり方 | 止まり方を知らず `max_new_tokens` まで書き続ける | 答え終えると `<|im_end|>` を出してピタッと止まる |

</details>

#### 📝 確認問題（`ch17-q02` · Lv.2 · ベースモデルが質問に答えず続きを書き続ける理由）

ベースモデル（`Qwen2.5-0.5B`）に「日本で一番高い山は何ですか？」とそのまま入力したところ、答えずに同じ質問文を繰り返したり、答えた後に勝手に次の「問：」を書き始めたりしました。なぜこのような挙動になるのですか？

- **a.** ベースモデルにはBPEトークナイザーが付属しておらず、日本語の質問文を正しくID列へ変換できないから
- **b.** `do_sample=False` を指定するとデコーダが同じ語句を無限反復するバグがHugging Faceにあるから
- **c.** ベースモデルは日本の地理や一般常識に関する知識を事前学習データに一切含んでおらず答えを知らないから
- **d.** 事前学習で文章の続きを書くことだけを学び、質問に答えて `<|im_end|>` で止まるSFTを経ていないから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：事前学習だけのベースモデルは、Web上の大量の文章から「次に来そうな文字列」を予測するように訓練されています。
- **ヒント 2**：Webの問題集やFAQページでは、1つの質問文や答えの直後に何が書かれていることが多いか想像してみましょう。
- **正解**：`d`

**考え方**：ベースモデルは純粋な「次トークン予測器」なので、質問文を見たら「よくある質問一覧の続き」を生成し、Q&A形式を見たら「次の問題」まで続けてしまいます。

1. 事前学習だけのベースモデルは、大量のWebテキストの続きを再現するのが仕事。
1. 質問に1回だけ答えて `<|im_end|>` で止まる作法は、SFT（第18章）で初めて身につける。
1. 会話アプリに使うときは `-Instruct` / `-Chat` の付いたチャットモデルを選ぶ。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q02

</details>

#### 📝 確認問題（`ch17-q03` · Lv.2 · チャットモデルができるまでの3段階（事前学習 → SFT → 好みの学習））

現代のチャット用LLMが作られる標準的な**3段階の順序と役割**として正しいものはどれですか？

- **a.** ①好みの学習（RLHF/DPOで安全性調整）→ ②SFT（会話作法）→ ③事前学習（最後に一般知識を詰め込む）
- **b.** ①事前学習（大量Webテキストのみ）で会話と安全性まで完結し、SFTや好みの学習は原則として不要である
- **c.** ①SFT（少数の対話例から学習開始）→ ②事前学習（Webテキスト追加）→ ③トークナイザーの語彙学習
- **d.** ①事前学習（知識・言語能力）→ ②SFT（会話作法と停止）→ ③好みの学習（RLHF/DPOで良さ・安全性を調整）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「膨大な一般教養を身につける → 接客研修で対話作法を学ぶ → フィードバックで良さや安全性を磨く」という流れに対応します。
- **ヒント 2**：第1段階がベースモデルづくり、第2段階が教師ありファインチューニング、第3段階が選好（Preference）アラインメントです。
- **正解**：`d`

**考え方**：第4部までで学んだ事前学習に続き、第5部では第18〜19章で **SFT / LoRA**、第22章で **DPO（好みの学習）** を順番に実践します。

1. 第1段階：事前学習（Pre-training）→ ベースモデル完成。
1. 第2段階：教師ありファインチューニング（SFT）→ 指示応答と停止を習得。
1. 第3段階：好みの学習（RLHF / DPO）→ 丁寧さ・正確さ・安全性をアラインメント。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q03

</details>

*(この節の追加問題 `ch17-q04` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html で自動採点・復習できます)*

## チャットテンプレートと複数ターンの会話

第1章で見た台本形式を、もう少し詳しく見ておきます。会話を続けるには、**これまでの会話を全部** `messages` に入れて渡します。LLMは、前回の会話を覚えていません。毎回、台本の最初から読み直しているのです。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
messages = [
    {"role": "system", "content": "あなたは親切なアシスタントです。"},
    {"role": "user", "content": "私の名前はミナトです。"},
    {"role": "assistant", "content": "こんにちは、ミナトさん！"},
    {"role": "user", "content": "私の名前は何でしたっけ？"},
]
print(chat_tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
<|im_start|>system
あなたは親切なアシスタントです。<|im_end|>
<|im_start|>user
私の名前はミナトです。<|im_end|>
<|im_start|>assistant
こんにちは、ミナトさん！<|im_end|>
<|im_start|>user
私の名前は何でしたっけ？<|im_end|>
<|im_start|>assistant
```

</details>

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(ask(chat_tok, chat, messages))

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
あなたの名前はミナトさんです。
```

</details>

> **💡 理論のポイント：「記憶」の正体**
>
> ChatGPTなどのアプリが会話を覚えているように見えるのは、アプリが**過去の会話を毎回まとめてLLMに送っている**からです。会話が長くなると、その分トークンが増え、計算が重くなり（第16章のKVキャッシュ）、やがてコンテキスト長（第2章）の上限に達します。長い会話で、最初のほうの話を忘れてしまうのはこのためです。

<details><summary><b>基礎の補足：LLMはステートレス！複数ターン `messages` の仕組みと履歴の切り詰め</b>（クリックで開く）</summary>

LLMの推論呼び出しは1回ごとに記憶がリセットされます（**ステートレス**）。アプリ側では毎ターン次のように `messages` リストを育てて送り直します。

```python no-run
# 1ターン目：[system, user1] を送る → assistant1 を受け取って追加
# 2ターン目：[system, user1, assistant1, user2] を全部送る → assistant2 を追加
# 3ターン目：履歴が長すぎたら、`system` は先頭に絶対残しつつ、古い (user, assistant) から捨てる！
sys_msgs = [m for m in messages if m["role"] == "system"]
dialog   = [m for m in messages if m["role"] != "system"]
trimmed  = sys_msgs + dialog[-2 * max_turns:]   # 直近 max_turns 往復だけ保持
```

「`system` メッセージを保護しながら直近の往復だけ残す」のが、実務のチャットアプリでコンテキスト長オーバーを防ぐ基本パターンです。

</details>

#### 📝 確認問題（`ch17-q05` · Lv.2 · LLMはステートレス：複数ターン会話の「記憶」の正体とコンテキスト上限）

チャットアプリで「私の名前はミナトです」と伝えた後、次のターンで「私の名前は何でしたっけ？」と聞くと「ミナトさんです」と答えてくれます。LLMが前の会話を覚えている**本当の仕組み**はどれですか？

- **a.** トークナイザーが自動的にユーザー情報をローカルディスクへ保存し、次の入力時にこっそり裏で挿入しているから
- **b.** LLM自体はステートレスで記憶を持たず、アプリ側が過去履歴をすべて `messages` に連結して毎回送るから
- **c.** ユーザーが発言するたびにモデルの重みパラメータが自動で追加学習され、名前などの情報が重みに書き込まれるから
- **d.** Transformerモデルの内部にセッションごとの無限容量データベースが内蔵されており、過去の発言を常時検索できるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：推論中のLLMの重みが会話ごとに書き換わるか、それとも毎回同じ固定関数として動いているか考えてみましょう。
- **ヒント 2**：本文のコードで、2ターン目に `apply_chat_template` へ渡していた `messages` リストに何が入っていたか思い出しましょう。
- **正解**：`b`

**考え方**：LLMの推論呼び出しは1回ごとに独立（ステートレス）です。会話の継続性は、過去の `user` / `assistant` 履歴を毎回台本として渡し直すことで実現しています。

1. 1ターン目：`[system, user1]` を送る → `assistant1` を受け取る。
1. 2ターン目：`[system, user1, assistant1, user2]` を**全部**送る → `assistant2` を受け取る。
1. 履歴が伸びるほど入力トークン数が増え、コンテキスト長の上限に達したら古い履歴を切り詰める必要がある。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q05

</details>

#### 📝 確認問題（`ch17-q06` · Lv.3 · 複数ターン会話での messages リスト構築と要素数の追跡）

複数ターン会話で `messages` リストにやり取りを追加していく処理を確認します。次のコードを実行すると何が表示されますか？

```python
messages = [{"role": "system", "content": "簡潔に答えて。"}]
for q, a in [("名前は？", "ボンゴです"), ("好きな食べ物は？", "バナナです")]:
    messages.append({"role": "user", "content": q})
    messages.append({"role": "assistant", "content": a})
messages.append({"role": "user", "content": "もう一度名前は？"})
print(len(messages), messages[-1]["role"])
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`messages` の初期要素数、ループ1回あたりに `append` される件数、ループ後に追加される要素を順に数えます。
- **ヒント 2**：最初に `system` が1件あり、2往復のループで `user` と `assistant` が2組（計4件）追加され、最後に新しい `user` が1件足されます。合計件数と末尾の `role` を答えましょう。
- **期待される出力**：`6 user`

**考え方**：1 + 2×2 + 1 = 6件になり、モデルへ応答生成を依頼する直前は必ず末尾が `user` メッセージになります（その直後に `add_generation_prompt=True` が `<|im_start|>assistant\n` を付け足します）。

1. 初期：`system`（1件）。
1. 2往復追加：`user`, `assistant`, `user`, `assistant`（+4件 → 計5件）。
1. 3ターン目の質問を追加：`user`（+1件 → 計6件）。
1. 長さと末尾のrole → `6 user`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q06

</details>

*(この節の追加問題 `ch17-q07` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html で自動採点・復習できます)*

## プロンプトの工夫と、その限界

モデルの重みを変えずに、**入力の書き方（プロンプト）だけで**性能を上げる工夫を、プロンプトエンジニアリングと呼びます。小さな実験で効果を確かめてみましょう。お題は「レビューが肯定的か否定的かを判定する」です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
reviews = [
    ("このラーメン、スープが最高においしかった！", "肯定"),
    ("店員さんの態度が悪くて二度と行かない。", "否定"),
    ("値段のわりに量が少なくてがっかり。", "否定"),
    ("駅から近くて便利、また来たいです。", "肯定"),
    ("注文してから40分も待たされた。", "否定"),
    ("期待していなかったけど、意外とよかった。", "肯定"),
]

def classify(review, few_shot=False):
    msgs = [{"role": "system", "content": "レビューが肯定的なら「肯定」、否定的なら「否定」とだけ答えてください。"}]
    if few_shot:   # 例を2つ見せる
        msgs += [{"role": "user", "content": "最高の一日でした。"}, {"role": "assistant", "content": "肯定"},
                 {"role": "user", "content": "最悪の対応でした。"}, {"role": "assistant", "content": "否定"}]
    msgs.append({"role": "user", "content": review})
    return ask(chat_tok, chat, msgs, n=5).strip()

for few in [False, True]:
    correct = sum(classify(r, few) == label for r, label in reviews)
    print(("few-shot " if few else "zero-shot"), f"正解 {correct}/{len(reviews)}")

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
zero-shot 正解 4/6
few-shot  正解 6/6
```

</details>

> **💡 理論のポイント：プロンプト技法**
>
> - **zero-shot**：お手本なしで、指示だけを与える
> - **few-shot**：答え方のお手本（例）を数個見せてから聞く。小さなモデルほど、出力の形式がそろって効果が大きい
> - **段階的に考えさせる**（Chain-of-Thought）：「ステップごとに考えてから答えてください」と頼むと、計算や推論の問題で正答率が上がることがある。答えの前に考える過程を書かせることで、その過程が次のトークンの手がかりになるため（第1章：LLMは自分の出力を読みながら次を予測する）
> - **役割（system）**：第1章のボンゴ先生のように、立場や口調を指定する
>
> あなたの結果はどうでしたか？ 上の出力は一例で、モデルの版などによって、正解数は変わります。few-shotのほうが良くなることが多いですが、同じだったり、逆になったりすることもあります。
>
> ただし、プロンプトには限界があります。どちらの結果が出たとしても、**6問は少なすぎて、偶然の差かもしれません**。本当に効果があるかを確かめるには、もっと多くの問題と、統計の考え方が必要です（第20章）。また、小さなモデルに知らない知識はプロンプトでは教えられませんし、毎回お手本を入れるとトークン（お金と時間）を食います。そこで、モデル自体を鍛える **ファインチューニング**（第18〜19章）や、資料を検索して渡す **RAG**（第21章）の出番になります。

<details><summary><b>基礎の補足：使い分けの地図：few-shotプロンプト vs ファインチューニング（第18〜19章） vs RAG（第21章）</b>（クリックで開く）</summary>

| 手法 | 重みを変える？ | 得意なこと | 限界・注意点 |
|---|---|---|---|
| **プロンプト（zero/few-shot）** | 変えない | すぐ試せる・出力形式を軽く整える | 毎回お手本分のトークンを消費／未知の専門知識は教えきれない／少問題では偶然に左右される |
| **ファインチューニング（SFT / LoRA）** | **変える**（第18〜19章） | 独自の口調・専門フォーマット・特定タスクの精度を定着させ、入力トークンも節約できる | 良質な学習データとGPU計算が必要／最新事実の更新には不向き |
| **RAG（検索拡張生成）** | 変えない（第21章） | 社内資料や毎日変わる最新情報を根拠つきで参照する | 検索精度が低いと誤った資料を渡してしまう |

</details>

> **🔬 やってみよう：段階的に考えさせる**
>
> `ask(chat_tok, chat, [{"role": "user", "content": "りんごが3個入った箱が4箱あり、2個食べました。残りは何個？"}])` と、同じ質問の最後に「ステップごとに考えてから答えてください。」を付けたものを比べてみましょう。小さなモデルなので、どちらも間違えるかもしれません。それも大事な観察結果です。

#### 📝 確認問題（`ch17-q08` · Lv.2 · zero-shot・few-shot・Chain-of-Thought（段階的思考）の違いと仕組み）

プロンプトに「ステップごとに考えてから答えてください」と付け加える手法（**Chain-of-Thought / CoT**）によって、算数や論理問題の正答率が上がることがある**本質的な理由**は何ですか？

- **a.** 思考過程を分割して出力するほうが総生成トークン数が大幅に減り、注意の分散が抑えられて速く解けるから
- **b.** 「ステップごとに」と指示すると、Transformer内部に組み込まれたPython電卓エンジンが自動起動するから
- **c.** 段階的思考を促す特別なプロンプトを入れると、推論時だけモデルの重みパラメータが一時的に2倍に増えるから
- **d.** 自分が生成した途中経過を読み直して次を予測するため、書き出した中間ステップが作業メモになるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：人間が暗算だけでいきなり答えを書く場合と、計算用紙に途中式を書きながら解く場合の違いを自己回帰モデルに当てはめてみましょう。
- **ヒント 2**：LLMは1トークンごとに一定量の計算しかできません。途中式をトークンとして出力すると、次のトークン予測の入力文脈に何が加わるでしょうか？
- **正解**：`d`

**考え方**：LLMは1トークンごとに一定量の計算（層数分の順伝播）しかできません。途中経過をテキストとして出力することで、複数トークン分の計算資源を使って段階的に答えへ近づけます。

1. いきなり答えを出す場合：1トークン分の順伝播で全計算を暗算しなければならない。
1. CoTで途中式を書く場合：各ステップの出力が次の入力文脈に入る。
1. その結果、小さな計算を連鎖させて正しい最終回答にたどり着きやすくなる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q08

</details>

#### 📝 確認問題（`ch17-q09` · Lv.3 · プロンプトの限界とファインチューニング／RAGの使い分け）

本文の実験では、6問のレビューで zero-shot が 4/6、few-shot が 6/6 でした。この結果の解釈とプロンプトの限界について**正しい**ものはどれですか？

- **a.** few-shotプロンプトは重みを永続的に書き換えるため、1回成功すれば次回からお手本を消しても同じ精度が出る
- **b.** 6問は少なすぎて偶然の差かもしれず十分な問題数が必要で、プロンプトにはトークン消費や知識追加の限界がある
- **c.** プロンプトへ100万件の社内文書を毎回貼り付ければ、ファインチューニングもRAGも永遠に不要になる
- **d.** 6問すべてに正解（6/6）できた時点で、未知の全レビューでも100%完璧に分類できると統計的に証明された

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「6問というサンプル数の少なさ」と「プロンプトだけでできること・できないこと」の2点から判断しましょう。
- **ヒント 2**：プロンプトは毎回入力トークンを消費し、モデルが知らない膨大な最新知識を詰め込むのには限界があります。
- **正解**：`b`

**考え方**：プロンプトエンジニアリングは最も手軽な第一歩ですが、①毎回トークンを食う、②最新・社内知識は教えきれない、③複雑な振る舞いの安定化に限界がある、という壁があるため、RAGやファインチューニングへ進みます。

1. 手軽な出力調整 → まずプロンプト（zero-shot / few-shot）。
1. 最新の外部知識・社内資料を正確に参照したい → **RAG（第21章）**。
1. 独自の口調・専門フォーマット・特定タスクの精度をトークン節約しつつ定着させたい → **ファインチューニング（第18〜19章）**。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q09

</details>

## 生成の設定を確かめる

モデルには、作った人がおすすめする生成の設定が付いてきます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(chat.generation_config)

<details><summary><b>📊 出力の例（数値や文章は実行ごとに変わってOK）</b>（実行後に自分の出力と見比べよう）</summary>

```text
GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "repetition_penalty": 1.1,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}
```

</details>

温度0.7、top-p 0.8、top-k 20、くり返しペナルティ1.1。第3章と第16章で学んだつまみが、全部出てきました。`eos_token_id` の151645は `<|im_end|>`（第2章）。これが出たら生成を止める、という設定です。

#### 📝 確認問題（`ch17-q10` · Lv.2 · generation_config の eos_token_id と do_sample=False の警告）

`chat.generation_config` を表示すると `"eos_token_id": [151645, 151643]`、`"temperature": 0.7`、`"top_p": 0.8` などが設定されていました。このうち **`eos_token_id`** の役割は何ですか？

- **a.** すべてのプロンプトの先頭に自動で挿入され、会話開始をモデルへ伝える挨拶トークンのID
- **b.** 不適切な表現を防ぐために、生成中の確率を強制的に0へ固定して出力を禁止する語句のID
- **c.** このID（例：`<|im_end|>`）を生成した時点で、上限未満でも生成ループを終了する合図
- **d.** 生成中にGPUメモリが逼迫した際、KVキャッシュを自動で4ビット量子化する際のしきい値

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`eos` は **End Of Sequence**（系列の終わり）の頭文字です。
- **ヒント 2**：モデルが答え終えたときに `<|im_end|>` を出力すると、`generate()` のループはどう反応するでしょうか？
- **正解**：`c`

**考え方**：チャットモデルは答え終えると `<|im_end|>`（Qwen2.5ではID `151645`）を出力します。`generate()` は `eos_token_id` が出た瞬間にループを終了します。

1. `eos_token_id` ＝ 終了合図のトークンID一覧。
1. 生成中に `151645`（`<|im_end|>`）が出たら即座に生成を止める。
1. ベースモデルが止まらなかったのは、答え終わりにこのトークンを出す訓練（SFT）を受けていないから。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q10

</details>

## モデルの大きさとGPUメモリ

借りるモデルを選ぶとき、一番の制約はGPUのメモリです。**推論（使うだけ）** に必要なメモリは、だいたい「パラメータ数 × 1個のバイト数」です。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def infer_gb(n_params_billion, bytes_per_param):
    return n_params_billion * bytes_per_param

print("        float16   8ビット   4ビット")
for size in [0.5, 1.5, 3, 7, 14]:
    print(f"{size:4}B  {infer_gb(size, 2):6.2f}GB {infer_gb(size, 1):6.2f}GB {infer_gb(size, 0.5):6.2f}GB")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
float16   8ビット   4ビット
 0.5B    1.00GB   0.50GB   0.25GB
 1.5B    3.00GB   1.50GB   0.75GB
   3B    6.00GB   3.00GB   1.50GB
   7B   14.00GB   7.00GB   3.50GB
  14B   28.00GB  14.00GB   7.00GB
```

</details>

T4（約15GB）なら、float16で3B程度まで余裕を持って動かせます。7Bはfloat16だとギリギリ（実際には計算用のメモリも必要なので厳しい）ですが、**4ビットに量子化**すれば3.5GBで動きます。量子化は第19章で学びます。

> **💡 理論のポイント：safetensors**
>
> モデルの重みのファイルには `model.safetensors` という形式がよく使われます。昔よく使われていた `pytorch_model.bin`（Pythonの `pickle` 形式）は、読み込むときに**任意のプログラムが実行される**危険がありました。知らない人が公開したファイルを開いたら、ウイルスが動く、ということが理論上ありえたのです。`safetensors` は数の並びだけを保存する形式なので、そうした危険がありません。知らない人のモデルを使うときは、safetensors形式を選びましょう。

<details><summary><b>基礎の補足：推論VRAM vs 学習VRAM：同じモデルでも必要メモリが「約8倍」違う！</b>（クリックで開く）</summary>

第15章の学習メモリ見積もり（混合精度＋AdamW＝1パラメータ約16バイト）と、この章の推論メモリ見積もりを並べて見比べてみましょう。

| モデル規模 | 推論（4ビット：0.5B/param） | 推論（float16：2B/param） | 全パラメータ学習（AdamW：16B/param） | T4（15GB）での可否 |
|---|---|---|---|---|
| **0.5B** | 0.25 GB | 1.0 GB | **8.0 GB**（＋活性化） | 推論も全学習もOK（第18章） |
| **3B** | 1.5 GB | 6.0 GB | **48.0 GB**（＋活性化） | float16推論は余裕、全学習は不可 |
| **7B** | 3.5 GB | 14.0 GB | **112.0 GB**（＋活性化） | 4ビット推論やQLoRA（第19章）ならOK！ |

推論では重み（＋KVキャッシュ）だけで済むのに対し、全パラメータ学習では fp16重み・fp32控え・勾配・AdamWの2つの状態・活性化を全部載せるため、約8倍以上のVRAMが必要です。

</details>

#### 📝 確認問題（`ch17-q11` · Lv.3 · 推論時の重みVRAM見積もり（float16 / 8ビット / 4ビット））

パラメータ数 **7B（70億）** のモデルを **4ビット量子化**（1パラメータあたり **0.5バイト**）で読み込むとき、モデルの重みだけでおよそ何GBのGPUメモリ（VRAM）を使いますか？（小数で答えてください）

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：推論時の重みVRAM（GB）はおおよそ「パラメータ数（B）× 1パラメータあたりのバイト数」で見積もれます。
- **ヒント 2**：4ビット量子化は $4 \div 8 = 0.5$ バイト/パラメータです。これを `7`（B）に掛けましょう。
- **正解**：`3.5` GB

**考え方**：$7 \times 0.5 = 3.5$ GB になります。これにKVキャッシュや計算用ワークスペースを足しても、無料T4 GPU（約15GB）で十分動かせます。

1. 4ビット ＝ $4 / 8 = 0.5$ バイト / パラメータ。
1. $7\text{B} \times 0.5\text{バイト} = 3.5$ GB。
1. float16（14GB）ではT4にギリギリだが、4ビット量子化なら3.5GBに収まる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q11

</details>

#### 📝 確認問題（`ch17-q12` · Lv.3 · 推論VRAM vs 学習VRAMの違い、および safetensors の安全性）

同じ **3B（30億パラメータ）** のモデルについて、**float16での推論**と**混合精度＋AdamWでの全パラメータ学習**を比べたとき、必要なGPUメモリの見積もりとして正しいものはどれですか？

- **a.** 推論の重みは約6GBでT4に乗るが、全パラメータ学習は勾配・Adam状態込みで約48GBとなりT4に乗らない
- **b.** 推論のほうがKVキャッシュ込みで約48GB必要になり、学習のほうは勾配圧縮により約6GBと省メモリになる
- **c.** 推論も全パラメータ学習も同じ3Bモデルの重みを使うため、どちらも約6GBでT4に余裕をもって収まる
- **d.** 3B規模なら推論も全パラメータ学習も1GB未満に収まるため、ノートPCやスマホでも余裕で学習できる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：推論（float16）と全パラメータ学習（混合精度＋AdamW）では、1パラメータあたりの必要バイト数が大きく異なります。
- **ヒント 2**：推論は2バイト/param、全パラメータ学習は重み・勾配・Adam状態を含めて約16バイト/param（約8倍）です。3Bに当てはめて比べましょう。
- **正解**：`a`

**考え方**：「推論できるサイズ」と「全パラメータ学習できるサイズ」には約8倍以上の開きがあります。このギャップを埋めるのが第19章の **QLoRA（4ビット量子化＋LoRA）** です。

1. 推論（float16）：$3\text{B} \times 2\text{B} = 6$ GB（＋KVキャッシュ）→ T4（15GB）で余裕。
1. 全パラメータ学習（AdamW）：$3\text{B} \times 16\text{B} = 48$ GB（＋活性化）→ T4ではOOM（メモリ不足）。
1. だから無料GPUで3B〜7Bを鍛えるには、全パラメータでなく一部だけ学習するLoRAが必要になる。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q12

</details>

## Hugging Faceのアカウントとトークン

第18章以降で、自分で鍛えたモデルをHugging Faceに保存したり、利用規約への同意が必要なモデルを使ったりするには、**アカウント**と**アクセストークン**が必要です。

1. <https://huggingface.co/join> で無料アカウントを作る（メールアドレスの確認まで）
2. 右上のアイコン →「Settings」→「Access Tokens」→「Create new token」
3. 種類（Token type）は、最初は **Read**（読み取り専用）を選ぶ。モデルをアップロードするときだけ **Write** の権限があるものを作る
4. 表示されたトークン（`hf_` で始まる文字列）をコピー

> **⚠️ つまずき注意：トークンは「家の鍵」**
>
> アクセストークンは、あなたのアカウントの鍵です。**ノートブックのコードに直接書いてはいけません**。ノートブックを人に見せたり、GitHubに上げたりしたときに、鍵ごと渡してしまうからです。Colabでは、左側の **鍵のアイコン（シークレット）** に `HF_TOKEN` という名前で保存し、「ノートブックからのアクセス」をオンにします。すると、コードに書かなくても、自動で使われるようになります（第1章で出た黄色い警告が消えます）。もし漏らしてしまったら、すぐにHugging Faceの設定画面でそのトークンを削除（Invalidate）しましょう。

コードからトークンを使う必要があるときは、次のように取り出します（シークレットが設定されていないとエラーになります）。

🌐 **オンライン接続セル**：Hugging Faceへの接続（必要に応じてトークン設定）が必要です。

🔎 **実行前に確認**：コードの各行が何を受け取り、何を作るか確認してから ▶ を押してください。

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

> **¥0 無料枠のコツ：無料でできること**
>
> Hugging Faceの無料アカウントで、公開モデル・データセットのダウンロード、自分のモデル・データセットの公開（公開リポジトリ）ができます。アプリを公開する「Spaces」は、無料アカウントだと使える種類が限られます（2026年10月時点の公式ドキュメントでは、CPUで動くGradioアプリの新規作成には有料プランが必要で、無料では「ZeroGPU」のSpaceが2つまで、または静的ページのSpaceが使えます）。第24章では、0円で確実に公開できる方法を紹介します。

> **⚠️ つまずき注意：よくあるつまずき**
>
> - **`OSError: … is not a local folder and is not a valid model identifier`**：モデル名のつづりが違うか、利用規約への同意が必要なモデル（gated model）です。Hugging Faceのモデルページで「Agree and access」を押し、`HF_TOKEN` を設定しましょう。
> - **`CUDA out of memory`**：モデルを2つ同時に読み込んでいるとメモリを食います。使わなくなったモデルは `del base` で消し、`torch.cuda.empty_cache()` を実行しましょう。
> - **ベースモデルが同じ文をくり返す**：正常です（この章の実験のとおり）。会話させたいならチャットモデル（Instruct）を使いましょう。
> - **`The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']`** という英語の表示：エラーではなく、ただのお知らせです。`do_sample=False`（サイコロを振らない）にしたので、モデル付属の設定にある温度などが使われない、と言っているだけです。無視してOKです。

#### 📝 確認問題（`ch17-q13` · Lv.2 · アクセストークンの安全な管理（シークレット・最小権限・漏洩時の対処））

Hugging Faceのアクセストークン（`hf_...`）について、①なぜノートブックのコードに直接書いてはいけないのか、②Colabではどのように安全に管理し、最初はどの権限（Read / Write）を選ぶべきか、③万が一漏洩してしまったら何をすべきか、の3点を説明してください。

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：①直書きによる流出経路、②Colabでの保管機能と最小権限の選び方、③漏洩発生時の即応手順、の3点に分けて整理しましょう。
- **ヒント 2**：①ノートブック共有やGitHub公開での漏洩リスク、②Colabシークレット（`HF_TOKEN`）と初期のRead権限、③Hugging Face設定画面でのトークン削除（無効化）を具体的に書きましょう。
- **解答の目安**：①アクセストークンは自分のアカウントの鍵であり、コードに直書きするとノートブックの共有やGitHub公開時に他人へ鍵ごと渡してしまい、なりすましや不正操作につながるためです。②Colabの左メニューにある鍵アイコン（シークレット）に `HF_TOKEN` として保存し、最初は最小権限の Read（読み取り専用）を選び、モデルのアップロードが必要なときだけ Write 権限のトークンを作ります。③万が一漏洩した場合は、直ちにHugging Faceの設定画面（Access Tokens）からそのトークンを削除（Invalidate / Revoke）して無効化します。

**考え方**：APIキーやアクセストークンの管理は、AI開発だけでなくソフトウェア開発全般で必須のセキュリティ基礎です。

1. 直書きNG：GitHubに1回でもpushすると履歴に永久に残り、ボットが数秒で悪用する。
1. 安全な管理：Colabシークレット（環境変数）＋必要最小限の権限（Read / Write分離）。
1. 漏洩時：コードを書き直すだけでなく、必ず発行元の設定画面でトークン自体を無効化する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html#ex-ch17-q13

</details>

**✅ できたらチェック**

- [ ] モデルカードで、説明・ライセンス・大きさを確認した
- [ ] ベースモデルとチャットモデルの違いを実験で確かめた
- [ ] 複数ターンの会話を `messages` で渡せた
- [ ] zero-shotとfew-shotを比べ、「6問では少なすぎる」理由を説明できる
- [ ] モデルの大きさから、必要なGPUメモリを見積もれた
- [ ] Hugging Faceのアカウントを作り、トークンをColabのシークレットに保存した

> **📝 まとめ**
>
> - チャットモデルは、**事前学習 → SFT（第18章）→ 好みの学習（第22章）** の3段階で作られる。ベースモデルは「続きを書く」だけ
> - 会話の「記憶」は、毎回これまでの会話全体を台本にして渡しているだけ
> - プロンプトの工夫（few-shotなど）は手軽だが、効果は**十分な数の問題で測る**必要があり、知識の追加には限界がある
> - 推論メモリ ≈ パラメータ数 × バイト数。T4ならfloat16で3B程度、4ビットなら7Bも動く
> - モデルの**ライセンス**を確認し、トークンは**シークレット**で管理する

**ボンゴ先生**：次の章では、いよいよ「接客研修」を自分でやる。ベースモデルに、質問に答えて止まる作法を教える **SFT** だ。まずは学習ループを自分で書いて仕組みを確かめてから、Qwenを「ボンゴ先生」に変身させよう。ダジャレの質は保証しない。

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/17-hf-models.html
- **次の章**：[第18章 指示に従わせる：SFT（教師ありファインチューニング）](https://sakunokyumei.github.io/llm-research-textbook/18-sft.html)